In [1]:
import pandas as pd


In [10]:
df = pd.read_csv('../datasets/dt_target.csv',sep='\t')
df.head()

,product_id,customer_id,periodo,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,cat1,cat2,cat3,brand,sku_size,descripcion,stock_final,periodo_dt,target
0,20524,10234,201701,0.0,2.0,0.05300,0.05300,HC,VAJILLA,Cristalino,Importado,500.0,Abrillantador,NaN,2017-01-01,0.01514
1,20524,10234,201702,NaN,NaN,NaN,0.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-02-01,0.00000
2,20524,10234,201703,0.0,1.0,0.01514,0.01514,HC,VAJILLA,Cristalino,Importado,500.0,Abrillantador,NaN,2017-03-01,0.00000
3,20524,10234,201704,NaN,NaN,NaN,0.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-04-01,0.03786
4,20524,10234,201705,NaN,NaN,NaN,0.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-05-01,0.00000


In [26]:
# Ordenar por producto, cliente y tiempo
df = df.sort_values(['product_id', 'customer_id', 'periodo_dt'])

df["periodo_dt"] = pd.to_datetime(df["periodo_dt"], format='%Y-%m-%d')
df["month"] = df["periodo_dt"].dt.month
df["year"] = df["periodo_dt"].dt.year
df['quarter'] = df['month'].apply(lambda x: (x-1)//3 + 1)
df['semester'] = df['month'].apply(lambda x: 1 if x <=6 else 2)
df['is_month_end'] = df['month'].isin([1, 3, 5, 7, 8, 10, 12])  # Meses con 31 días
df['season'] = df['month']%12 // 3 + 1  # 1:Invierno, 2:Primavera, etc.
df['size_vs_category'] = df['sku_size'] / df.groupby('cat3')['sku_size'].transform('mean')


# Crear lags
df['lag_1m'] = df.groupby(['product_id', 'customer_id'])['tn'].shift(1)
df['lag_2m'] = df.groupby(['product_id', 'customer_id'])['tn'].shift(2)
df['lag_3m'] = df.groupby(['product_id', 'customer_id'])['tn'].shift(3)

# Promedio móvil
df['rolling_3m_mean'] = df.groupby(['product_id', 'customer_id'])['tn'].transform(
    lambda x: x.rolling(3, min_periods=1).mean())


# Tendencia anual
df['annual_trend'] = df.groupby(['product_id', 'month'])['tn'].transform('mean')

# Variación estacional
df['seasonal_variation'] = df['tn'] / df['annual_trend']

In [23]:
df["descripcion"].unique()

array(['Abrillantador', nan, 'Sabor 8',
       'Claldo Granulado verduras sin sal', 'Claldo Granulado verduras',
       'Fideos al verdeo', 'Multiuso 1', 'Multiuso 3',
       'Caldo Granulado sin sal', 'Regular', 'Salsa golf', 'Mostaza',
       'Ketchup', 'paño tradicional', 'Almidon', 'Fideos Tirabuzon',
       'Albhaca', 'Cuatro Quesos', 'Exfoliante',
       'Crema rejuvenecimiento noche', 'Carnes Patagonicas',
       'Verduras multiples', 'Gallina', 'Salsa Aji Picante',
       'Fideos con salsa bolognesa', 'Fideo con berengenas',
       'Ketchup tomate', 'Salsa Barbacoa', 'Salsa Tomate y Carne',
       'Sabor 4', 'Sabor 1', 'Sabor 7', 'Sabor 6', 'Sesamo', 'Sabor 2',
       'Sabor 5', 'Chimichurri', 'Fideos con mix de vegetales', 'Blanca',
       'Hidrtacion', 'Polvo Maquina', 'BioEnergético doypack',
       'Estilo 16', 'Estilo 25', 'Sabor 13', 'Velocidad', 'Clasico',
       'Duluido clasico', 'Lima', 'Ultra Glicerina', 'Menta',
       'ropa negra', 'ropa blanca', 'Poderoso Baño', '

In [22]:
df["cat2"].unique()

array(['VAJILLA', nan, 'DEOS', 'SOPAS Y CALDOS', 'HOGAR', 'PROFESIONAL',
       'ADEREZOS', 'OTROS', 'PIEL2', 'PIEL1', 'ROPA ACONDICIONADOR',
       'ROPA LAVADO', 'CABELLO', 'ROPA MANCHAS', 'TE', 'DENTAL'],
      dtype=object)

In [25]:
print(df.head())

   product_id  customer_id  periodo  plan_precios_cuidados  cust_request_qty  \
0       20524        10234   201701                    0.0               2.0   
1       20524        10234   201702                    NaN               NaN   
2       20524        10234   201703                    0.0               1.0   
3       20524        10234   201704                    NaN               NaN   
4       20524        10234   201705                    NaN               NaN   

   cust_request_tn       tn cat1     cat2        cat3      brand  sku_size  \
0          0.05300  0.05300   HC  VAJILLA  Cristalino  Importado     500.0   
1              NaN  0.00000  NaN      NaN         NaN        NaN       NaN   
2          0.01514  0.01514   HC  VAJILLA  Cristalino  Importado     500.0   
3              NaN  0.00000  NaN      NaN         NaN        NaN       NaN   
4              NaN  0.00000  NaN      NaN         NaN        NaN       NaN   

     descripcion  stock_final periodo_dt   target 